# AnalysisExamples2

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `analysis`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/AnalysisExamples2.md`


Notebook source link: [AnalysisExamples2.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/AnalysisExamples2.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "AnalysisExamples2"
FAMILY = "analysis"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"AnalysisExamples2: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"AnalysisExamples2: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"AnalysisExamples2: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"AnalysisExamples2: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "warning off;",
    "installPath = which('nSTAT_Install');",
    "if isempty(installPath)",
    "error('AnalysisExamples2:MissingInstallPath', ...",
    "'Could not locate nSTAT_Install.m on the MATLAB path.');",
    "end",
    "glmDataPath = fullfile(fileparts(installPath), 'data', 'glm_data.mat');",
    "load(glmDataPath);",
    "nst = nspikeTrain(spiketimes);",
    "baseline = Covariate(T,ones(length(xN),1),'Baseline','time','s','',{'mu'});",
    "position = Covariate(T,[xN yN],'Position', 'time','s','m',{'x','y'});",
    "velocity = Covariate(T,[vxN,vyN],'Velocity','time','s','m/s',{'v_x','v_y'});",
    "radial   = Covariate(T,[xN yN xN.^2 yN.^2 xN.*yN],'Radial','time','s','m',{'x','y','x^2','y^2','x*y'});",
    "[values_at_spiketimes] =position.getValueAt(spiketimes);",
    "[values_at_spiketimes] =position.resample(1/min(diff(spiketimes))).getValueAt(spiketimes);",
    "figure;",
    "plot(position.getSubSignal('x').dataToMatrix,position.getSubSignal('y').dataToMatrix,...",
    "values_at_spiketimes(:,1),values_at_spiketimes(:,2),'r.');",
    "axis tight square;",
    "xlabel('x position (m)'); ylabel('y position (m)');",
    "spikeColl = nstColl({nst});",
    "covarColl = CovColl({baseline,radial});",
    "trial     = Trial(spikeColl,covarColl);",
    "clear tc;",
    "sampleRate=1000;",
    "tc{1} = TrialConfig({{'Baseline','mu'},{'Radial','x','y'}},sampleRate,[]); tc{1}.setName('Linear');",
    "tc{2} = TrialConfig({{'Baseline','mu'},{'Radial','x','y','x^2','y^2','x*y'}},sampleRate,[]); tc{2}.setName('Quadratic');",
    "tc{3} = TrialConfig({{'Baseline','mu'},{'Radial','x','y','x^2','y^2','x*y'}},sampleRate,[0 1]./sampleRate); tc{3}.setName('Quadratic+Hist');",
    "tcc = ConfigColl(tc); makePlot=1; neuronNum=1;",
    "fitResults =Analysis.RunAnalysisForAllNeurons(trial,tcc,0);",
    "fitResults.plotResults;",
    "figure;",
    "[x_new,y_new]=meshgrid(-1:.1:1); %define new x and y",
    "y_new = flipud(y_new);",
    "x_new = fliplr(x_new);",
    "newData{1} =ones(size(x_new));",
    "newData{2} =x_new; newData{3} =y_new;",
    "newData{4} =x_new.^2; newData{5} =y_new.^2;",
    "newData{6} =x_new.*y_new;",
    "color = Analysis.colors;",
    "for i=1:fitResults.numResults",
    "lambda = fitResults.evalLambda(i,newData);",
    "h_mesh = mesh(x_new,y_new,lambda,'AlphaData',0);",
    "get(h_mesh,'AlphaData');",
    "set(h_mesh,'FaceAlpha',0.2,'EdgeAlpha',0.8,'EdgeColor',color{i});",
    "hold on;",
    "end",
    "legend(fitResults.lambda.dataLabels);",
    "plot(position.getSubSignal('x').dataToMatrix,position.getSubSignal('y').dataToMatrix,...",
    "values_at_spiketimes(:,1),values_at_spiketimes(:,2),'r.');",
    "axis tight square;",
    "xlabel('x position (m)'); ylabel('y position (m)');",
    "[b,dev,stats] = glmfit([xN yN xN.^2 yN.^2 xN.*yN],spikes_binned,'poisson');",
    "b-fitResults.b{2} % should be close to zero",
    "sampleRate=1000;  makePlot=1; neuronNum = 1;",
    "covLabels = {{'Baseline','mu'},{'Radial','x','y','x^2','y^2','x*y'}};",
    "Algorithm = 'GLM';",
    "batchMode=0;",
    "windowTimes =(0:1:10)./sampleRate;",
    "[fitResults,tcc] = Analysis.computeHistLag(trial,neuronNum,windowTimes,covLabels,Algorithm,batchMode,sampleRate,makePlot);"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for AnalysisExamples2.")


In [ ]:
# AnalysisExamples2: compare linear and quadratic spatial Poisson GLMs.
n_t = 5000
dt = 0.01
time = np.arange(n_t) * dt

xy = np.zeros((n_t, 2), dtype=float)
xy[0] = np.array([50.0, 50.0])
vel = np.zeros(2, dtype=float)
for t in range(1, n_t):
    vel = 0.9 * vel + 2.4 * rng.normal(size=2)
    xy[t] = np.clip(xy[t - 1] + vel, 0.0, 100.0)

xc, yc, sigma = 35.0, 70.0, 14.0
r2 = (xy[:, 0] - xc) ** 2 + (xy[:, 1] - yc) ** 2
true_rate = 1.0 + 20.0 * np.exp(-0.5 * r2 / (sigma**2))
counts = rng.poisson(true_rate * dt)

X_lin = np.column_stack([xy[:, 0], xy[:, 1]])
X_quad = np.column_stack([xy[:, 0], xy[:, 1], xy[:, 0] ** 2, xy[:, 1] ** 2, xy[:, 0] * xy[:, 1]])

fit_lin = Analysis.fit_glm(X=X_lin, y=counts, fit_type="poisson", dt=dt)
fit_quad = Analysis.fit_glm(X=X_quad, y=counts, fit_type="poisson", dt=dt)

grid = np.linspace(0.0, 100.0, 35)
gx, gy = np.meshgrid(grid, grid, indexing="xy")
Xg_lin = np.column_stack([gx.ravel(), gy.ravel()])
Xg_quad = np.column_stack([gx.ravel(), gy.ravel(), gx.ravel() ** 2, gy.ravel() ** 2, gx.ravel() * gy.ravel()])
true_map = 1.0 + 20.0 * np.exp(
    -0.5 * (((Xg_lin[:, 0] - xc) ** 2 + (Xg_lin[:, 1] - yc) ** 2) / (sigma**2))
)
lin_map = fit_lin.predict(Xg_lin)
quad_map = fit_quad.predict(Xg_quad)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
im0 = axes[0, 0].imshow(true_map.reshape(grid.size, grid.size), origin="lower", extent=[0, 100, 0, 100], cmap="jet")
axes[0, 0].set_title("True field")
fig.colorbar(im0, ax=axes[0, 0], fraction=0.04, pad=0.03)

im1 = axes[0, 1].imshow(lin_map.reshape(grid.size, grid.size), origin="lower", extent=[0, 100, 0, 100], cmap="jet")
axes[0, 1].set_title("Linear GLM field")
fig.colorbar(im1, ax=axes[0, 1], fraction=0.04, pad=0.03)

im2 = axes[1, 0].imshow(quad_map.reshape(grid.size, grid.size), origin="lower", extent=[0, 100, 0, 100], cmap="jet")
axes[1, 0].set_title("Quadratic GLM field")
fig.colorbar(im2, ax=axes[1, 0], fraction=0.04, pad=0.03)

aic_vals = np.array([fit_lin.aic(), fit_quad.aic()], dtype=float)
axes[1, 1].bar(["linear", "quadratic"], aic_vals, color=["tab:gray", "tab:blue"])
axes[1, 1].set_title("Model comparison (AIC)")
axes[1, 1].set_ylabel("AIC")

plt.tight_layout()
plt.show()

rmse_lin = float(np.sqrt(np.mean((fit_lin.predict(X_lin) - true_rate) ** 2)))
rmse_quad = float(np.sqrt(np.mean((fit_quad.predict(X_quad) - true_rate) ** 2)))
print("rmse_lin", rmse_lin, "rmse_quad", rmse_quad)
assert rmse_quad <= rmse_lin + 0.8

CHECKPOINT_METRICS = {
    "rmse_lin": float(rmse_lin),
    "rmse_quad": float(rmse_quad),
}
CHECKPOINT_LIMITS = {
    "rmse_lin": (0.0, 20.0),
    "rmse_quad": (0.0, 20.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
